In [0]:
checkpoint = "/Volumes/ev_spark/myvol/checkpoint/chkpt/bc_ev_silver/"

In [0]:
def readBronzeBcEv():
    print("Beginning to read from bronze")
    from pyspark.sql import functions as F
    readBcEv_df = (spark.readStream
                    .table("ev_spark.bronze.bc_ev")
                    .withColumn("ev_count", F.regexp_replace(F.col("year_2025"), ",", "").cast("int"))
                    .withColumn("province", F.lit("BC"))
                    .select("city", "province", "ev_count")
    )
    print("Read successful")
    print("*************************")
    return readBcEv_df

In [0]:
def writeToSilver(df):
    print("Beginning to write to silver")
    (df.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint)
        .outputMode("append")
        .trigger(availableNow = True)
        .toTable("ev_spark.silver.bc_ev")
    )
    print("Write successful")
    print("*************************")
    

In [0]:
readBcEv_df = readBronzeBcEv()
writeToSilver(readBcEv_df)

In [0]:
%sql
select * from ev_spark.silver.bc_ev limit 5